# Dimensiones SCD en Azure

Inicialización y aplicación controlada de cambios en productos y clientes,
reutilizando las funciones probadas localmente.

Las vigencias representan estados observados.
Las tablas originales de M9 se conservan.

Ejecutar las celdas en orden en Azure Databricks.
Ante un fallo parcial, revisar el informe y los destinos antes de repetir.
Las celdas de configuración generan una nueva carpeta de demostración.

In [ ]:
from pathlib import Path
import json
import sys
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F

current = Path.cwd().resolve()
repo_root = next(
    (
        path for path in (current, *current.parents)
        if (path / "transformations" / "gold").is_dir()
        and (path / "sources" / "reference").is_dir()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("No se encontró la raíz del repositorio.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from transformations.gold.scd_initialization import initialize_scd_dimension
from transformations.gold.scd_transform import (
    SCD_RULES,
    prepare_scd_dimension,
    validate_scd_history,
)
from transformations.gold.scd_storage import (
    read_scd_dimension,
    save_scd_dimension,
)
from transformations.gold.scd_sales_keys import validate_legacy_sales
from transformations.gold.storage import write_verified_gold_table

spark.conf.set("spark.sql.session.timeZone", "UTC")
print("PASS: funciones SCD importadas; sesión UTC.")


## Configuración

La ejecución M9 de origen es explícita.
Cada demostración SCD tendrá una carpeta independiente.

Conservar estos identificadores durante la ejecución.
Ante un fallo parcial, revisar los destinos antes de repetir escrituras.

In [ ]:
gold_root = "abfss://gold@starlpdevdiego01.dfs.core.windows.net"
source_run_id = "953a045c-f46b-4f1a-bb98-5003f14b6930"
source_report_path = (
    f"{gold_root}/_executions/run_id={source_run_id}/report.json"
)

scd_run_id = str(uuid4())
scd_root = f"{gold_root}/scd/run_id={scd_run_id}"
targets = {
    name: f"{scd_root}/tables/{name}"
    for name in ("dim_product", "dim_customer")
}

print("Informe M9:", source_report_path)
print("Ejecución SCD:", scd_run_id)
for name, target in targets.items():
    print(name, target)


## Lectura de la base aceptada

Se utilizan las rutas y versiones registradas en el informe M9.
Se comprueban estado, conteos y procedencia de las tres tablas utilizadas.

Esta lectura no reconstruye Gold ni modifica sus destinos.

In [ ]:
source_report = json.loads(
    dbutils.fs.head(source_report_path, 1024 * 1024)
)

if (
    source_report["scope"] != "azure_gold_initial"
    or source_report["status"] != "completed"
    or source_report["gold_processing_run_id"] != source_run_id
):
    raise ValueError("El informe no corresponde a la base M9 aceptada.")

expected_counts = {
    "dim_product": 61,
    "dim_customer": 500,
    "fact_sales": 5862,
}
accepted = {}
source_records = {}

for name, expected_count in expected_counts.items():
    record = source_report["tables"][name]
    if record["verified"] is not True:
        raise ValueError(f"Tabla sin escritura verificada: {name}")
    if record["row_count"] != expected_count:
        raise ValueError(f"Conteo de referencia inesperado: {name}")

    frame = read_scd_dimension(
        spark, record["path"], record["version"]
    )
    if frame.count() != expected_count:
        raise ValueError(f"Conteo persistido diferente: {name}")

    if frame.filter(
        ~F.col("gold_processing_run_id").eqNullSafe(F.lit(source_run_id))
    ).limit(1).count():
        raise ValueError(f"Procedencia diferente a la ejecución M9: {name}")

    accepted[name] = frame
    source_records[name] = dict(record)
    print(
        f"PASS: {name}, versión={record['version']}, "
        f"filas={expected_count}"
    )


## Inicialización SCD

Crear las copias iniciales preservando claves, atributos y procedencia M9.

Todas las filas comienzan con valid_from igual a la observación inicial,
valid_to nulo e is_current verdadero.

Las escrituras son exclusivas: no sobrescriben tablas existentes.
Si ocurre un fallo parcial, revisar el informe y los destinos.
No repetir esta sección automáticamente.

In [ ]:
if globals().get("_scd_initialization_started", False):
    raise RuntimeError(
        "La inicialización ya comenzó; revisar los destinos antes de repetir."
    )

initial_observed_at = datetime.now(timezone.utc)
initial_processed_at = initial_observed_at
initial_batch_id = f"{scd_run_id}-initial"

initial_frames = {
    name: initialize_scd_dimension(
        accepted[name],
        name,
        initial_batch_id,
        scd_run_id,
        initial_observed_at,
        initial_processed_at,
    )
    for name in targets
}

for name, frame in initial_frames.items():
    validate_scd_history(frame, name)

report_path = f"{scd_root}/report.json"
report = {
    "scope": "azure_scd_demonstration",
    "status": "initializing",
    "scd_run_id": scd_run_id,
    "source_gold_processing_run_id": source_run_id,
    "source_report_path": source_report_path,
    "sources": source_records,
    "initialization": {
        "batch_id": initial_batch_id,
        "observed_at": initial_observed_at.isoformat(),
        "processed_at": initial_processed_at.isoformat(),
        "tables": {},
    },
    "batches": [],
}


def save_scd_report():
    dbutils.fs.put(
        report_path,
        json.dumps(report, ensure_ascii=False, indent=2) + "\n",
        overwrite=True,
    )


_scd_initialization_started = True
dbutils.fs.mkdirs(scd_root)
save_scd_report()

try:
    initialized = {}

    for name, frame in initial_frames.items():
        result = write_verified_gold_table(
            spark, frame, targets[name]
        )
        report["initialization"]["tables"][name] = result
        save_scd_report()

        recovered = read_scd_dimension(
            spark, result["path"], result["version"]
        )
        validate_scd_history(recovered, name)

        original = accepted[name]
        preserved = recovered.select(*original.columns)
        if (
            original.exceptAll(preserved).limit(1).count()
            or preserved.exceptAll(original).limit(1).count()
        ):
            raise ValueError(f"Se alteraron datos originales de M9: {name}")

        initialized[name] = recovered
        print(f"PASS: {name}, {result['row_count']} filas inicializadas.")

    legacy_result = validate_legacy_sales(
        accepted["fact_sales"],
        accepted["fact_sales"],
        {name: accepted[name] for name in targets},
        initialized,
    )
    report["initialization"]["legacy_sales"] = legacy_result
    report["status"] = "initialized"
    save_scd_report()

    print("PASS: copias SCD verificadas y referencias M9 conservadas.")
    print("Informe:", report_path)

except Exception as error:
    report["status"] = "failed"
    report["error"] = str(error)
    save_scd_report()
    raise


La comprobación anterior verifica que las ventas M9 siguen encontrando
las mismas entidades mediante sus claves originales en las copias SCD.

La tabla de hechos utilizada es la misma base aceptada;
esta comprobación no representa una comparación entre dos copias de ventas.
No se escriben ni se modifican ventas.

## Entradas de demostración

Entradas sintéticas construidas desde la base M9 leída anteriormente.
No se modifica Silver ni se presenta esta demostración como ingesta real.

Por dimensión se seleccionan tres entidades existentes, ordenadas por
clave de negocio: corrección tipo 1, cambio tipo 2 y cambio combinado.
Se añade una entidad nueva y las demás permanecen sin cambios.

El precio de catálogo no modifica los importes de ventas existentes.
Las entradas se preparan en memoria; este bloque no escribe tablas.

In [ ]:
if report["status"] != "initialized":
    raise RuntimeError("Se requiere la inicialización SCD verificada.")

incoming_batches = {}
expected_changes = {}

for name in targets:
    rules = SCD_RULES[name]
    business_key = rules["business_key"]
    attributes = rules["stable"] + rules["type1"] + rules["type2"]
    columns = (business_key,) + attributes
    baseline = accepted[name].select(*columns)

    selected_ids = [
        row[business_key]
        for row in baseline.orderBy(business_key).limit(3).collect()
    ]
    if len(selected_ids) != 3:
        raise ValueError("Se requieren tres entidades para la demostración.")

    type1_id, type2_id, combined_id = selected_ids
    maximum = baseline.agg(F.max(business_key)).first()[0]
    if maximum >= 2147483647:
        raise ValueError("No hay espacio para una nueva clave de negocio.")
    new_id = maximum + 1

    descriptive = "product_name" if name == "dim_product" else "first_name"
    type1_condition = F.col(business_key).isin(type1_id, combined_id)
    type2_condition = F.col(business_key).isin(type2_id, combined_id)

    replacements = {
        descriptive: F.when(
            type1_condition,
            F.concat(
                F.coalesce(F.col(descriptive), F.lit("")),
                F.lit(" [corregido SCD]"),
            ),
        ).otherwise(F.col(descriptive)),
    }

    if name == "dim_product":
        price_type = baseline.schema["list_price"].dataType
        replacements["list_price"] = F.when(
            type2_condition,
            (F.col("list_price") + F.lit(1)).cast(price_type),
        ).otherwise(F.col("list_price"))
    else:
        for column, value in (
            ("city", "Ciudad demostración"),
            ("region", "Región demostración"),
        ):
            replacements[column] = F.when(
                type2_condition, F.lit(value)
            ).otherwise(F.col(column))

    updated = baseline.select(*[
        replacements.get(column, F.col(column)).alias(column)
        for column in columns
    ])

    new_values = {
        business_key: F.lit(new_id).cast("int"),
        descriptive: F.lit("Entidad nueva SCD"),
        "is_active": F.lit(True),
    }
    if name == "dim_product":
        new_values["sku"] = F.lit(f"SCD-{scd_run_id}")

    new_entity = (
        baseline.filter(F.col(business_key) == type1_id)
        .select(*[
            new_values.get(column, F.col(column)).alias(column)
            for column in columns
        ])
    )

    incoming_batches[name] = updated.unionByName(new_entity)
    expected_changes[name] = {
        "type1": 1,
        "type2": 1,
        "type1_and_type2": 1,
        "new": 1,
        "unchanged": expected_counts[name] - 3,
    }
    print(
        f"{name}: tipo 1={type1_id}, tipo 2={type2_id}, "
        f"combinado={combined_id}, alta={new_id}"
    )

print("PASS: entradas sintéticas preparadas sin escrituras.")


## Aplicación del lote

Guardar las entradas sintéticas como Delta y leer su versión verificada.
Aplicar correcciones, cierres e inserciones mediante un MERGE por dimensión.

Se esperan tres filas adicionales por dimensión:
dos nuevas versiones de entidades existentes y una entidad nueva.

Productos y clientes tienen commits independientes.
Ante un fallo parcial, revisar el informe antes de repetir la ejecución.

In [ ]:
if report["status"] != "initialized":
    raise RuntimeError("El lote requiere destinos inicializados sin cambios.")

batch_id = f"{scd_run_id}-demo-1"
batch_run_id = str(uuid4())
batch_observed_at = datetime.now(timezone.utc)
batch_processed_at = batch_observed_at

if batch_observed_at <= initial_observed_at:
    raise ValueError("La observación debe ser posterior a la inicialización.")

batch_record = {
    "batch_id": batch_id,
    "run_id": batch_run_id,
    "input_kind": "synthetic_demonstration",
    "observed_at": batch_observed_at.isoformat(),
    "processed_at": batch_processed_at.isoformat(),
    "status": "running",
    "inputs": {},
    "dimensions": {},
}
report["batches"].append(batch_record)
report["status"] = "applying"
save_scd_report()

try:
    stored_inputs = {}

    for name, frame in incoming_batches.items():
        input_path = f"{scd_root}/inputs/demo-1/{name}"
        input_record = write_verified_gold_table(spark, frame, input_path)
        batch_record["inputs"][name] = input_record
        save_scd_report()

        stored_inputs[name] = read_scd_dimension(
            spark, input_record["path"], input_record["version"]
        )

    applied = {}

    for name in targets:
        initial_record = report["initialization"]["tables"][name]
        base_version = initial_record["version"]
        dimension = read_scd_dimension(
            spark, targets[name], base_version
        )

        prepared = prepare_scd_dimension(
            stored_inputs[name],
            dimension,
            name,
            batch_id,
            batch_run_id,
            batch_observed_at,
            batch_processed_at,
        )

        actual_changes = {
            row["change_type"]: row["count"]
            for row in prepared["changes"].groupBy("change_type").count().collect()
        }
        if actual_changes != expected_changes[name]:
            raise ValueError(
                f"Clasificación inesperada en {name}: {actual_changes}"
            )

        batch_record["dimensions"][name] = {
            "base_version": base_version,
            "changes": actual_changes,
        }
        save_scd_report()

        result = save_scd_dimension(
            prepared["dimension"], targets[name], name, base_version
        )
        batch_record["dimensions"][name]["result"] = result
        save_scd_report()

        if (
            result["changed"] is not True
            or result["version"] != base_version + 1
            or result["row_count"] != expected_counts[name] + 3
        ):
            raise ValueError(f"Resultado inesperado en {name}: {result}")

        recovered = read_scd_dimension(
            spark, result["path"], result["version"]
        )
        validate_scd_history(recovered, name)

        current_count = recovered.filter(F.col("is_current")).count()
        if current_count != expected_counts[name] + 1:
            raise ValueError(f"Cantidad de entidades actuales inesperada: {name}")

        applied[name] = recovered
        batch_record["dimensions"][name]["history_verified"] = True
        batch_record["dimensions"][name]["current_count"] = current_count
        save_scd_report()

        print(
            f"PASS: {name}, versión={result['version']}, "
            f"filas={result['row_count']}, actuales={current_count}"
        )

    batch_record["legacy_sales"] = validate_legacy_sales(
        accepted["fact_sales"],
        accepted["fact_sales"],
        {name: accepted[name] for name in targets},
        applied,
    )
    batch_record["status"] = "completed"
    report["status"] = "applied"
    save_scd_report()

    print("PASS: lote aplicado, historial y referencias M9 verificados.")
    print("Informe:", report_path)

except Exception as error:
    batch_record["status"] = "failed"
    batch_record["error"] = str(error)
    report["status"] = "failed"
    save_scd_report()
    raise


## Repetición sin cambios

Volver a preparar las mismas entradas contra los destinos actualizados.

Se conserva la observación del lote y se utiliza otro identificador de intento.
Todas las entidades deben quedar sin cambios y Delta debe conservar su versión.

Esta prueba demuestra igualdad con el estado actual.
No implica reconocimiento persistente de lotes históricos ni recuperación
automática ante fallos.

In [ ]:
if report["status"] != "applied":
    raise RuntimeError("Se requiere el lote aplicado y verificado.")

repeat_run_id = str(uuid4())
repeat_processed_at = datetime.now(timezone.utc)

repetition = {
    "run_id": repeat_run_id,
    "processed_at": repeat_processed_at.isoformat(),
    "status": "running",
    "dimensions": {},
}
report["repetition"] = repetition
report["status"] = "checking_repetition"
save_scd_report()

try:
    for name in targets:
        input_record = batch_record["inputs"][name]
        previous_result = batch_record["dimensions"][name]["result"]
        base_version = previous_result["version"]

        incoming = read_scd_dimension(
            spark, input_record["path"], input_record["version"]
        )
        current = read_scd_dimension(
            spark, targets[name], base_version
        )

        prepared = prepare_scd_dimension(
            incoming,
            current,
            name,
            batch_id,
            repeat_run_id,
            batch_observed_at,
            repeat_processed_at,
        )

        changes = {
            row["change_type"]: row["count"]
            for row in prepared["changes"].groupBy("change_type").count().collect()
        }
        if changes != {"unchanged": expected_counts[name] + 1}:
            raise ValueError(f"La entrada repetida contiene cambios: {name}")

        result = save_scd_dimension(
            prepared["dimension"], targets[name], name, base_version
        )
        repetition["dimensions"][name] = {
            "changes": changes,
            "result": result,
        }
        save_scd_report()

        if (
            result["changed"] is not False
            or result["version"] != base_version
            or result["row_count"] != previous_result["row_count"]
            or result["verified"] is not True
        ):
            raise ValueError(f"La repetición modificó el destino: {name}")

        print(
            f"PASS: {name}, sin cambios; "
            f"versión={result['version']}, filas={result['row_count']}"
        )

    repetition["status"] = "completed"
    report["status"] = "completed"
    report["finished_at"] = datetime.now(timezone.utc).isoformat()
    save_scd_report()

    print("PASS: demostración SCD en Azure completada.")
    print("Informe:", report_path)
    print(json.dumps(report, ensure_ascii=False, indent=2))

except Exception as error:
    repetition["status"] = "failed"
    repetition["error"] = str(error)
    report["status"] = "failed"
    save_scd_report()
    raise
